In [1]:
import uuid, time
from src.spark.spark import get_spark_session
from src.utils.logging_config import setup_logging
from src.io.writer import DataFrameWriter

In [ ]:
layer = "gold"
dataset = "fact_product_performance"

# Load Spark Session
spark = get_spark_session()

# Load logging
log = setup_logging(f"{dataset}_{layer}")

# Create logical variables
run_id = str(uuid.uuid4())

In [ ]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")

start = time.time()
log.info(f"[GOLD][START] run_id={run_id}")

try:

    writer = DataFrameWriter(spark)
    
    fact_sales = spark.sql("""
        WITH reviews AS(
            SELECT 
                orr.order_id,
                AVG(orr.review_score) AS avg_review_score
            FROM olist.silver.order_reviews orr 
            GROUP BY orr.order_id
        )
        SELECT
            p.product_id,
            p.product_category_name,
            p.product_weight_g,
            p.product_length_cm,
            p.product_height_cm,
            p.product_width_cm,
            oi.order_id,
            oi.freight_value,
            r.avg_review_score
        FROM olist.silver.order_items oi
        LEFT JOIN olist.silver.products p ON p.product_id = oi.product_id
        LEFT JOIN reviews r ON r.order_id = oi.order_id
    """)
    
    writer.write_delta_batch(fact_sales, "olist", "gold", "fact_product_performance", mode="overwrite")
    duration_s = round(time.time() - start, 3)
    log.info(f"[GOLD][SUCCESS] run_id={run_id} dataset={dataset} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[GOLD][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise